In [1]:
# ===============================================================
# CSIRO Image2Biomass – Training (Weighted R² Validation)
# ===============================================================
import os, gc, cv2, numpy as np, pandas as pd
from tqdm import tqdm
import torch, torch.nn as nn, torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torch.nn.parameter import Parameter
import torch.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold
from torch.amp import GradScaler, autocast
import wandb
from datetime import datetime
from sklearn.model_selection import StratifiedGroupKFold
import random
# ---------------------------------------------------------------
# 1. CONFIG (memory-safe + R² metric)
# ---------------------------------------------------------------
class CFG:
    BASE_PATH       = 'csiro-biomass'
    TRAIN_CSV       = os.path.join(BASE_PATH, 'train.csv')
    TRAIN_IMAGE_DIR = os.path.join(BASE_PATH, 'train')
    MODEL_DIR       = 'out'
    N_FOLDS         = 5

    MODEL_NAME = 'convnext_base'      # safe & matches inference
    CHECKPOINT_PATH = None #'lora_finetuned_convnext_base.pt'
    IMG_SIZE   = 512                  # fits easily
    PRETRAINED = True
    FREEZE_BACKBONE = False

    BATCH_SIZE   = 1
    GRAD_ACC     = 8                  # effective batch = 8
    NUM_WORKERS  = 1
    EPOCHS       = 100
    LR           = 1e-4
    WD           = 0.01 #1e-2 convnext
    PATIENCE     = 10
    WARMUP_EPOCHS = 2
    WARMUP_HEAD_EPOCHS = 5

    DETERMINISTIC = True
    SEED = 694

    TARGET_COLS    = ['Dry_Total_g', 'GDM_g', 'Dry_Green_g']
    DERIVED_COLS   = ['Dry_Clover_g', 'Dry_Dead_g']
    ALL_TARGET_COLS = ['Dry_Green_g','Dry_Dead_g','Dry_Clover_g','GDM_g','Dry_Total_g']
    R2_WEIGHTS     = np.array([0.1, 0.1, 0.1, 0.2, 0.5])  # matches metric
    W_SPECIES = 0.25
    W_STATE = 0.25
    W_CONT = 0.5

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device : {CFG.DEVICE}")
print(f"Backbone: {CFG.MODEL_NAME} | Size: {CFG.IMG_SIZE}")

def set_seed(seed=42, deterministic=True):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# ---------------------------------------------------------------
# 2. WEIGHTED R² METRIC (your function)
# ---------------------------------------------------------------
def global_weighted_r2_score(y_true: np.ndarray, y_pred: np.ndarray):
    """
    Computes the globally weighted R² score as described in the evaluation.
    
    y_true, y_pred: shape (N, 5)
    weights: [0.1, 0.1, 0.1, 0.2, 0.5] (from CFG)
    """
    weights_matrix = np.tile(CFG.R2_WEIGHTS, (y_true.shape[0], 1))
    # y_bar_w = (sum(w_j * y_j)) / (sum(w_j))
    weighted_sum = np.sum(weights_matrix * y_true)
    total_weight = np.sum(weights_matrix)
    y_bar_w = weighted_sum / total_weight # This is a single scalar value
    # SS_res = sum(w_j * (y_j - y_pred_j)^2)
    ss_res = np.sum(weights_matrix * (y_true - y_pred) ** 2)
    # SS_tot = sum(w_j * (y_j - y_bar_w)^2)
    ss_tot = np.sum(weights_matrix * (y_true - y_bar_w) ** 2)
    # R²_w = 1 - (SS_res / SS_tot)
    r2_w = 1 - (ss_res / ss_tot)
    return r2_w
def weighted_r2_score(y_true: np.ndarray, y_pred: np.ndarray):
    """
    y_true, y_pred: shape (N, 5)
    weights: [0.1, 0.1, 0.1, 0.2, 0.5]
    """
    weights = CFG.R2_WEIGHTS
    r2_scores = []
    for i in range(5):
        y_t = y_true[:, i]
        y_p = y_pred[:, i]
        ss_res = np.sum((y_t - y_p) ** 2)
        ss_tot = np.sum((y_t - np.mean(y_t)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        r2_scores.append(r2)
    r2_scores = np.array(r2_scores)
    weighted_r2 = np.sum(r2_scores * weights) / np.sum(weights)
    return weighted_r2
# ---------------------------------------------------------------
# 3. AUGMENTATIONS
# ---------------------------------------------------------------
def get_spatial_transforms():
    # These will be applied to BOTH images identically
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        # A.PadIfNeeded(
        #     min_height=CFG.IMG_SIZE, # Set this to 1024
        #     min_width=CFG.IMG_SIZE,  # Set this to 1024
        #     border_mode=cv2.BORDER_REFLECT_101 # Best padding mode
        # ),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
    ], 
    p=1.0,
    additional_targets={'image_right': 'image'},
    seed=CFG.SEED 
    )
def get_photometric_transforms():
    # These will be applied INDEPENDENTLY to each half
    return A.Compose([
        A.ColorJitter(brightness=0.5,contrast=0.5,saturation=0.5,hue=0.0,p=0.5),
        # A.GaussianBlur(blur_limit=(0, 2), p=0.3),
        # A.GaussNoise(std_range=(0,0.1),mean_range=(0,0),p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std =[0.229, 0.224, 0.225]),
        ToTensorV2()
    ], p=1.0, seed=CFG.SEED)

def get_train_transforms():
    # This single transform will be applied to EACH of the 8 patches
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE), # CFG.IMG_SIZE must be 336
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ColorJitter(
            brightness=0.5,
            contrast=0.5,
            saturation=0.4,
            hue=0.0,
            p=0.5
        ),
        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std =[0.229, 0.224, 0.225]),
        ToTensorV2()
    ], p=1.0)

def get_val_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE), # CFG.IMG_SIZE must be 336
        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std =[0.229, 0.224, 0.225]),
        ToTensorV2()
    ], p=1.0)
# ---------------------------------------------------------------
# X. Discriminative learning rate
# ---------------------------------------------------------------
def create_optimizer_groups(model, lr_heads, lr_backbone):
    """
    Splits model parameters into 'backbone' and 'heads' groups.
    'heads' includes everything that is NOT the backbone.
    """
    parameters = [
        # Group 1: Backbone
        {
            "params": [p for n, p in model.named_parameters() if "backbone" in n],
            "lr": lr_backbone,
            "name": "Backbone"
        },
        # Group 2: Heads (everything else)
        {
            "params": [p for n, p in model.named_parameters() if "backbone" not in n],
            "lr": lr_heads,
            "name": "Heads"
        }
    ]
    
    # Check if the 'Heads' group is empty (which would be a bug)
    if not parameters[1]["params"]:
        print("Warning: 'Heads' group has 0 parameters.")
        
    return parameters
# ---------------------------------------------------------------
# 4. DATASET
# ---------------------------------------------------------------
class BiomassDataset(Dataset):
    def __init__(self, df, transform, photometric_transform, img_dir):
        self.df        = df
        self.transform = transform
        self.ph_transform = photometric_transform
        self.img_dir   = img_dir
        self.paths     = df['image_path'].values
        self.labels    = df[CFG.ALL_TARGET_COLS].values.astype(np.float32)
        self.aux_cat_labels = df[CFG.CAT_AUX_COLS].values.astype(np.int64)
        self.aux_cont_labels = df[CFG.CONT_AUX_COLS].values.astype(np.float32)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        path = os.path.join(self.img_dir, os.path.basename(self.paths[idx]))
        img  = cv2.imread(path)
        if img is None:
            img = np.zeros((1000, 2000, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        h, w, _ = img.shape
        mid = w // 2
        left  = img[:, :mid]
        right = img[:, mid:]
        if self.transform:
            transformed = self.transform(image=left, image_right=right)
            left  = transformed['image']
            right = transformed['image_right']

        # 2. Apply PHOTOMETRIC transforms (independently)
        left  = self.ph_transform(image=left)['image']
        right = self.ph_transform(image=right)['image']

        label = torch.from_numpy(self.labels[idx])
        aux_cat_label   = torch.from_numpy(self.aux_cat_labels[idx])
        aux_cont_label  = torch.from_numpy(self.aux_cont_labels[idx])
        return left, right, label, aux_cat_label, aux_cont_label

# ---------------------------------------------------------------
# 5. MODEL (safe pretrained loading)
# ---------------------------------------------------------------
class ContinuousMetadataGate(nn.Module):
    def __init__(self, n_cont_features, image_feat_dim):
        super().__init__()
        self.image_feat_dim = image_feat_dim
        
        self.mlp = nn.Sequential(
            nn.Linear(n_cont_features, 128),
            nn.LayerNorm(128),
            nn.ReLU(inplace=True),
            # nn.Dropout(0.2), # Remove dropout in the gate, we want stability
            nn.Linear(128, image_feat_dim * 2) 
        )

    def forward(self, aux_cont):
        # 1. HARD INPUT CLIPPING
        # Prevent outliers (e.g., > 3 std devs) from exploding the gate
        aux_safe = torch.clamp(aux_cont, -3.0, 3.0)
        
        out = self.mlp(aux_safe)
        
        gamma_raw, beta_raw = torch.split(out, self.image_feat_dim, dim=1)
        
        # 2. DAMPENING (The "Soft" Gate)
        # Tanh forces range [-1, 1].
        # Multiply by 0.1 to limit effect to +/- 10%.
        gamma = torch.tanh(gamma_raw) * 0.5
        
        # Beta can be larger, but let's keep it controlled too
        beta = beta_raw * 0.5
        
        return gamma, beta
    
def gem(x, p=3, eps=1e-6):
    return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1./p)
class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super(GeM,self).__init__()
        self.p = Parameter(torch.ones(1)*p)
        self.eps = eps
    def forward(self, x):
        return gem(x, p=self.p, eps=self.eps)       
    def __repr__(self):
        return self.__class__.__name__ + '(' + 'p=' + '{:.4f}'.format(self.p.data.tolist()[0]) + ', ' + 'eps=' + str(self.eps) + ')'

class BiomassModel(nn.Module):
    def __init__(self, model_name, n_species, n_states, n_cont_targets, pretrained=True, freeze_backbone=False,checkpoint_path=None):
        super().__init__()
        
        # 1. Image Backbone
        self.backbone = timm.create_model(
            model_name, pretrained=False, num_classes=0, global_pool='avg')
        
        if checkpoint_path:
            # SIMPLE LOADING
            weights = torch.load(checkpoint_path, map_location='cpu')
            # strict=False allows ignoring the 'head' layers if dimensions differ
            self.backbone.load_state_dict(weights, strict=False)

        # self.backbone.avg_pool=GeM()
        nf = self.backbone.num_features
        
        # We have TWO image feature streams (left + right)
        image_feature_dim = nf * 2
        
        # self.meta_gate = ContinuousMetadataGate(
        #     n_cont_features=n_cont_targets, 
        #     image_feat_dim=image_feature_dim
        # )

        # 3. Main Head
        self.head = nn.Sequential(
            nn.Linear(image_feature_dim, image_feature_dim//2), 
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(image_feature_dim//2, image_feature_dim//4),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )
        self.regressor = nn.Linear(image_feature_dim//4, 3)

        # 4. Aux Head (Optional, acts on raw image features)
        self.head_height = nn.Sequential(
            nn.Linear(image_feature_dim, image_feature_dim//2), 
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(image_feature_dim//2, image_feature_dim//4),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(image_feature_dim//4, 2)
        )

        if pretrained:
            self.load_pretrained()
        if freeze_backbone:
            self.freeze_backbone()

    def load_pretrained(self):
        try:
            # Note: Ensure CFG is accessible or pass model_name here
            state_dict = timm.create_model(self.backbone.default_cfg['architecture'], pretrained=True, num_classes=0).state_dict()
            self.backbone.load_state_dict(state_dict, strict=False)
            print("Pretrained weights loaded (CPU)")
        except Exception as e:
            print(f"Warning: Pretrained load failed: {e}")

    def freeze_backbone(self):
        print("Freezing backbone parameters.")
        for param in self.backbone.parameters():
            param.requires_grad = False
            
    def unfreeze_backbone(self):
        print("Unfreezing backbone parameters.")
        for param in self.backbone.parameters():
            param.requires_grad = True

    def forward(self, left, right, aux_cat, aux_cont=None):
        # 1. Extract Raw Image Features
        fl = self.backbone(left)
        fr = self.backbone(right)
        image_features = torch.cat([fl, fr], dim=1) # [B, 1536] (if ConvNeXt-Tiny)

        aux_preds = self.head_height(image_features)

        # if self.training and aux_cont is not None:
        #     # TEACHER FORCING (50/50 split)
        #     if torch.rand(1).item() < 0.2:
        #         meta_input = aux_cont
        #     else:
        #         meta_input = aux_preds.detach() 
        # else:
        #     # INFERENCE
        #     meta_input = aux_preds

        # gamma, beta = self.meta_gate(meta_input)

        # modulated_features = image_features * (1 + gamma) + beta
        safe_features = image_features# + modulated_features
        fused = self.head(safe_features)
        predictions = self.regressor(fused)
        
        p_total, p_gdm, p_green = predictions.split(1, dim=1)
        
        return (p_total, p_gdm, p_green), aux_preds

# ---------------------------------------------------------------
# 6. LOSS (MSE on all 5)
# ---------------------------------------------------------------
def weighted_biomass_loss(p_total, p_gdm, p_green, labels, use_huber=False):
    """
    Calculates the 5 individual MSE losses and returns their
    weighted sum, perfectly aligning with the R2 metric weights.
    """
    loss_fn = nn.HuberLoss(delta=15.0) if use_huber else nn.MSELoss()
    
    # 1. Calculate the 5 individual MSE losses
    loss_total = loss_fn(p_total.squeeze(), labels[:, 4]) # Corresponds to Dry_Total_g
    loss_gdm   = loss_fn(p_gdm.squeeze(),   labels[:, 3]) # Corresponds to GDM_g
    loss_green = loss_fn(p_green.squeeze(), labels[:, 0]) # Corresponds to Dry_Green_g

    # Calculate derived predictions
    p_clover = torch.clamp(p_gdm - p_green, min=0)
    p_dead   = torch.clamp(p_total - p_gdm, min=0)

    loss_clover = loss_fn(p_clover.squeeze(), labels[:, 2]) # Corresponds to Dry_Clover_g
    loss_dead   = loss_fn(p_dead.squeeze(),   labels[:, 1]) # Corresponds to Dry_Dead_g

    # 2. Get the weights
    weights = CFG.R2_WEIGHTS
    
    # 3. Apply the weights to their corresponding losses
    weighted_loss_sum = (
        loss_green  * weights[0] +
        loss_dead   * weights[1] +
        loss_clover * weights[2] +
        loss_gdm    * weights[3] +
        loss_total  * weights[4]
    )
    
    return weighted_loss_sum

# ---------------------------------------------------------------
# 7. VALIDATION WITH WEIGHTED R²
# ---------------------------------------------------------------
@torch.no_grad()
def valid_epoch(model, loader, device):
    model.eval()
    running_loss = 0.0
    running_aux_loss = 0.0
    preds = {'total':[], 'gdm':[], 'green':[]}
    all_labels = []

    for l, r, lab, aux_cat, aux_cont in tqdm(loader, desc='valid', leave=False):
        l, r, lab = l.to(device, non_blocking=True), r.to(device, non_blocking=True), lab.to(device, non_blocking=True)
        aux_cont=aux_cont.to(device, non_blocking=True)
        aux_cat=aux_cat.to(device, non_blocking=True)
        with autocast('cuda',dtype=torch.bfloat16):
            (p_tot, p_gdm, p_green), p_aux = model(l, r, aux_cat, aux_cont=None)

            loss = weighted_biomass_loss(p_tot, p_gdm, p_green, lab)
            loss_aux_diag = nn.MSELoss()(p_aux, aux_cont)

        running_loss += loss.item() * l.size(0)
        running_aux_loss += loss_aux_diag.item() * l.size(0)

        preds['total'].extend(p_tot.cpu().float().numpy().ravel())
        preds['gdm'].extend(p_gdm.cpu().float().numpy().ravel())
        preds['green'].extend(p_green.cpu().float().numpy().ravel())
        all_labels.extend(lab.cpu().float().numpy())

    # Convert to numpy
    pred_total = np.array(preds['total'])
    pred_gdm   = np.array(preds['gdm'])
    pred_green = np.array(preds['green'])
    true_labels = np.stack(all_labels)  # (N, 5)

    # Compute derived
    pred_clover = np.clip(pred_gdm - pred_green, 0, None)
    pred_dead   = np.clip(pred_total - pred_gdm, 0, None)

    # Stack predictions in correct order
    pred_all = np.stack([
        pred_green,      # Dry_Green_g
        pred_dead,       # Dry_Dead_g
        pred_clover,     # Dry_Clover_g
        pred_gdm,        # GDM_g
        pred_total       # Dry_Total_g
    ], axis=1)

    # Compute weighted R²
    weighted_r2 = global_weighted_r2_score(true_labels, pred_all)

    return running_loss / len(loader.dataset), running_aux_loss/len(loader.dataset), weighted_r2, pred_all, true_labels

# ---------------------------------------------------------------
# 8. TRAINING LOOP
# ---------------------------------------------------------------
loss_fn_categorical = nn.CrossEntropyLoss()
loss_fn_continuous = nn.MSELoss()        # MSE or L1Loss
def train_epoch(model, loader, opt, scheduler, device, scaler):
    model.train()
    running = 0.0

    opt.zero_grad()
    for i, (l, r, lab, aux_cat, aux_cont) in enumerate(tqdm(loader, desc='train', leave=False)):
        l, r, lab = l.to(device, non_blocking=True), r.to(device, non_blocking=True), lab.to(device, non_blocking=True)
        aux_cat = aux_cat.to(device, non_blocking=True)
        aux_cont = aux_cont.to(device, non_blocking=True)
        with autocast('cuda',dtype=torch.bfloat16):
        # p_tot, p_gdm, p_green, p_species, p_state, p_cont = model(l, r)
            (p_tot, p_gdm, p_green), p_aux = model(l, r, aux_cat, aux_cont)
            
            loss_reg = weighted_biomass_loss(p_tot, p_gdm, p_green, lab, use_huber=False)
            # loss_aux = nn.MSELoss()(p_aux, aux_cont)

            total_loss = loss_reg #+ (0.5 * loss_aux)
        
        loss = total_loss / CFG.GRAD_ACC
        scaler.scale(loss).backward()
        # loss.backward()
        running += loss.item() * l.size(0) * CFG.GRAD_ACC

        if (i + 1) % CFG.GRAD_ACC == 0 or (i + 1) == len(loader):
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            # opt.step()
            opt.zero_grad()

    scheduler.step()
    return running / len(loader.dataset)

/home/teo/kaggle/csiro/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cuda
Backbone: convnext_base | Size: 512


In [2]:
from configs.deterministic import *
from datetime import datetime
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# ---------------------------------------------------------------
# 9. MAIN – 5-FOLD WITH R² TRACKING
# ---------------------------------------------------------------
set_seed(CFG.SEED, CFG.DETERMINISTIC)

print("Loading data...")
df_long = pd.read_csv(CFG.TRAIN_CSV)
df_wide = df_long.pivot(index='image_path', columns='target_name', values='target').reset_index()
df_wide = df_wide[['image_path'] + CFG.ALL_TARGET_COLS]
print(f"{len(df_wide)} training images")

# Aux task
aux_cols = ['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm']
df_aux = df_long[aux_cols].drop_duplicates().reset_index(drop=True)

df_wide = df_wide.merge(df_aux, on='image_path', how='left')

df_wide['State_idx'],   STATE_MAP   = pd.factorize(df_wide['State'])
df_wide['Species_idx'], SPECIES_MAP = pd.factorize(df_wide['Species'])

CFG.N_STATES   = len(STATE_MAP)
CFG.N_SPECIES  = len(SPECIES_MAP)
print(f"Found {CFG.N_STATES} states and {CFG.N_SPECIES} species.")

# 2. Convert Date to cyclical features (we'll predict these)
df_wide['Sampling_Date'] = pd.to_datetime(df_wide['Sampling_Date'])
df_wide['day_of_year'] = df_wide['Sampling_Date'].dt.dayofyear
df_wide['day_sin'] = np.sin(2 * np.pi * df_wide['day_of_year'] / 365.25)
df_wide['day_cos'] = np.cos(2 * np.pi * df_wide['day_of_year'] / 365.25)

# 3. Define all continuous columns to be scaled
# We'll predict NDVI and Height, so they must be scaled as targets
CFG.CONT_AUX_COLS = ['Pre_GSHH_NDVI', 'Height_Ave_cm']#, 'day_sin', 'day_cos']
CFG.CAT_AUX_COLS  = ['State_idx', 'Species_idx']

group_name = f"Date_{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

config = {k: v for k, v in vars(CFG).items() if not k.startswith('_')}
df_wide['group'] = df_wide['State'].astype(str) + "_" + df_wide['Sampling_Date'].astype(str)
df_wide['biomass_bin'] = pd.qcut(df_wide['Dry_Total_g'], q=10, labels=False)
sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)

# Store the best R2 score from each fold
all_fold_best_scores = []

print(f"{'Fold':<5} | {'Train Count':<12} | {'Val Count':<10} | {'Val %':<8} | {'Val Mean Biomass':<18}")
print("-" * 65)

fold_stats = []

# 2. The Loop
for fold, (tr_idx, val_idx) in enumerate(sgkf.split(df_wide, df_wide['biomass_bin'], groups=df_wide['group'])):
    
    # Get the actual data for this fold
    train_fold = df_wide.iloc[tr_idx]
    val_fold   = df_wide.iloc[val_idx]
    
    # Calculate stats
    n_train = len(train_fold)
    n_val   = len(val_fold)
    ratio   = n_val / (n_train + n_val) * 100
    
    # Check "Hardness" (Mean target value)
    # If one fold has a mean of 100 and another 500, your folds are NOT balanced.
    val_mean = val_fold['Dry_Total_g'].mean()
    
    print(f"{fold+1:<5} | {n_train:<12} | {n_val:<10} | {ratio:<6.2f}% | {val_mean:<18.4f}")
    
    fold_stats.append(n_val)

# 3. Check Deviation
mean_size = np.mean(fold_stats)
max_dev = np.max(np.abs(fold_stats - mean_size)) / mean_size * 100
print("-" * 65)
print(f"Max deviation from ideal size: {max_dev:.2f}%")
print("Checking categorical column ranges:")
print(f"State_idx max: {df_wide['State_idx'].max()}, CFG.N_STATES: {CFG.N_STATES}")
print(f"Species_idx max: {df_wide['Species_idx'].max()}, CFG.N_SPECIES: {CFG.N_SPECIES}")
for fold, (tr_idx, val_idx) in enumerate(sgkf.split(X=df_wide, y=df_wide['biomass_bin'], groups=df_wide['group'])):
    print('\n' + '='*70)
    print(f'   FOLD {fold+1}/{CFG.N_FOLDS}   |   {len(tr_idx)} train / {len(val_idx)} val')
    print('='*70)
    # wandb.init(
    #         project="csiro-biomass",
    #         group=group_name,           # Group all folds under this name
    #         name=f"{group_name}_f{fold}", # Unique name for this fold
    #         config=config,              # Log config for each run
    #         reinit=True                 # Allow re-initialization
    #     )
    torch.cuda.empty_cache()
    gc.collect()
    print(val_idx)
    tr_df  = df_wide.iloc[tr_idx].reset_index(drop=True)
    val_df = df_wide.iloc[val_idx].reset_index(drop=True)

    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()

    # Fit on train, transform both
    tr_df[CFG.CONT_AUX_COLS] = scaler.fit_transform(tr_df[CFG.CONT_AUX_COLS])
    val_df[CFG.CONT_AUX_COLS] = scaler.transform(val_df[CFG.CONT_AUX_COLS])

    tr_set = BiomassDataset(tr_df, get_spatial_transforms(), get_photometric_transforms(), CFG.TRAIN_IMAGE_DIR)
    val_set= BiomassDataset(val_df, None, get_val_transforms(), CFG.TRAIN_IMAGE_DIR)

    def seed_worker(worker_id):
        s = torch.initial_seed() % 2**32
        np.random.seed(s)
        random.seed(s)
    g = torch.Generator()
    g.manual_seed(CFG.SEED)

    tr_loader  = DataLoader(tr_set,  batch_size=CFG.BATCH_SIZE, shuffle=True,
                            num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True, worker_init_fn=seed_worker,generator=g)
    val_loader = DataLoader(val_set, batch_size=CFG.BATCH_SIZE, shuffle=False,
                            num_workers=CFG.NUM_WORKERS, pin_memory=True, worker_init_fn=seed_worker,generator=g)

    print("Building model...")
    model = BiomassModel(
            CFG.MODEL_NAME, 
            n_species=0,  # Pass in class numbers
            n_states=0,
            n_cont_targets=0, # Pass in number of cont. targets
            pretrained=True, 
            freeze_backbone=False,
            checkpoint_path=CFG.CHECKPOINT_PATH
        )
    model = model.to(CFG.DEVICE)
    # model = nn.DataParallel(model)

    if CFG.FREEZE_BACKBONE:
        parameters = filter(lambda p: p.requires_grad, model.parameters())
    else:
        parameters = create_optimizer_groups(
            model=model,
            lr_heads=CFG.LR,
            lr_backbone=CFG.LR/100# fixed for the moment
        )

    optimizer = optim.AdamW(parameters, lr=CFG.LR, weight_decay=CFG.WD )

    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=1e-6, # Start from a very small LR
        end_factor=1.0,
        total_iters=CFG.WARMUP_EPOCHS
    )

    main_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=CFG.EPOCHS - CFG.WARMUP_EPOCHS
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, main_scheduler],
        milestones=[CFG.WARMUP_EPOCHS]
    )
    best_r2 = -np.inf
    patience = 0
    try:
        # wandb.watch(model, log="all", log_freq=100)
        scaler = GradScaler()
        for epoch in range(1, CFG.EPOCHS+1):
            tr_loss = train_epoch(model, tr_loader, optimizer, scheduler, CFG.DEVICE, scaler)
            val_loss, aux_loss, val_r2, _, _ = valid_epoch(model, val_loader, CFG.DEVICE)

            print(f'Epoch {epoch:02d} | '
                    f'TrainLoss {tr_loss:.5f} | '
                    f'ValLoss {val_loss:.5f} | '
                    f'AuxLoss {aux_loss:.5f} | '
                    f'ValR² {val_r2:.4f} {"(BEST)" if val_r2 > best_r2 else ""}')

            if val_r2 > best_r2:
                best_r2 = val_r2
                save_path = os.path.join(CFG.MODEL_DIR, f'best_model_fold{fold}.pth')
                torch.save(model.module.state_dict() if hasattr(model, 'module') else model.state_dict(), save_path)
                print(f'   → SAVED (R²: {best_r2:.4f})')
                patience = 0
            else:
                patience += 1
                if patience >= CFG.PATIENCE:
                    print(f'   → EARLY STOP (no improvement in {CFG.PATIENCE} epochs)')
                    # for e in range(epoch + 1, CFG.EPOCHS + 1):
                    #     wandb.log({
                    #         "train_loss": tr_loss, # Carry forward last known loss
                    #         "val_loss": val_loss,  # Carry forward last known loss
                    #         "val_r2": best_r2,     # CRITICAL: Carry forward the BEST score
                    #         "best_r2": best_r2,
                    #     }, step=e)
                    break
            log_data = {
                    "train_loss": tr_loss,
                    "val_loss": val_loss,
                    "val_r2": val_r2,
                    "best_r2":best_r2,
                }
                
            # wandb.log(log_data, step=epoch)
        
        all_fold_best_scores.append(best_r2)
    finally:
        # wandb.finish()
        pass
        
# Cleanup
del model, tr_loader, val_loader, optimizer, main_scheduler
torch.cuda.empty_cache()
gc.collect()
mean_cv_score = np.mean(all_fold_best_scores)
std_cv_score  = np.std(all_fold_best_scores)

print('\n' + '='*70)
print('         FINAL CROSS-VALIDATION SCORE')
print('='*70)
print(f'Public LB Score: 0.58')
print(f'Local CV Score: {mean_cv_score:.3f} ± {std_cv_score:.3f}')
print('\nIndividual Fold Scores:')
for i, score in enumerate(all_fold_best_scores):
    print(f'  Fold {i+1}: {score:.4f}')

import csv

run_message = "good folds, no height aux, mse for train, default augs"

log_file = os.path.join(CFG.MODEL_DIR, 'experiment_log.csv')
log_data = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'cv_mean': f"{mean_cv_score:.5f}",
    'cv_std': f"{std_cv_score:.5f}",
    'message': run_message,
    'model': CFG.MODEL_NAME,
    'lr': CFG.LR,
    'epochs': CFG.EPOCHS,
    'batch_size': CFG.BATCH_SIZE,
    'img_size': CFG.IMG_SIZE,
    'frozen': CFG.FREEZE_BACKBONE
}
fieldnames = list(log_data.keys())

# 3. WRITE TO THE CSV FILE
# Check if file exists to write header only once
file_exists = os.path.isfile(log_file)

try:
    with open(log_file, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()  # Write header if new file
        writer.writerow(log_data) # Append new run data
    
    print(f"\nExperiment results saved to {log_file}")
except Exception as e:
    print(f"\nError saving experiment log: {e}")

    print('\nTraining complete! Best models saved in:', CFG.MODEL_DIR)
    print('Use these in inference with:')
    print('   MODEL_NAME = "convnext_tiny"')
    print('   IMG_SIZE = 512')

Loading data...
357 training images
Found 4 states and 15 species.
Fold  | Train Count  | Val Count  | Val %    | Val Mean Biomass  
-----------------------------------------------------------------
1     | 285          | 72         | 20.17 % | 51.2549           
2     | 281          | 76         | 21.29 % | 53.6554           
3     | 286          | 71         | 19.89 % | 34.7921           
4     | 286          | 71         | 19.89 % | 44.2094           
5     | 290          | 67         | 18.77 % | 41.8103           
-----------------------------------------------------------------
Max deviation from ideal size: 6.44%
Checking categorical column ranges:
State_idx max: 3, CFG.N_STATES: 4
Species_idx max: 14, CFG.N_SPECIES: 15

   FOLD 1/5   |   285 train / 72 val
[  2   7   9  18  21  23  24  29  34  36  37  39  43  44  45  47  52  69
  70  82 103 110 113 114 117 118 125 129 131 132 140 147 151 156 162 165
 176 181 182 189 191 195 198 203 230 231 234 238 245 248 249 254 262 264
 267 27

train:   0%|          | 0/285 [00:00<?, ?it/s]/home/teo/kaggle/csiro/.venv/lib/python3.10/site-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 01 | TrainLoss 1799.95075 | ValLoss 2541.30980 | AuxLoss 0.92851 | ValR² -1.4449 (BEST)
   → SAVED (R²: -1.4449)


Epoch 02 | TrainLoss 1607.01599 | ValLoss 1924.21198 | AuxLoss 0.94005 | ValR² -0.8512 (BEST)
   → SAVED (R²: -0.8512)


Epoch 03 | TrainLoss 737.92115 | ValLoss 707.01184 | AuxLoss 0.92054 | ValR² 0.3198 (BEST)
   → SAVED (R²: 0.3198)


...

Epoch 07 | TrainLoss 245.93077 | ValLoss 628.77924 | AuxLoss 2.37140 | ValR² 0.4551 


Epoch 08 | TrainLoss 207.11708 | ValLoss 516.46316 | AuxLoss 2.35695 | ValR² 0.5524 (BEST)
   → SAVED (R²: 0.5524)


Epoch 09 | TrainLoss 225.57264 | ValLoss 429.56537 | AuxLoss 2.36808 | ValR² 0.6277 (BEST)
   → SAVED (R²: 0.6277)


Epoch 10 | TrainLoss 157.30863 | ValLoss 371.94609 | AuxLoss 2.39446 | ValR² 0.6777 (BEST)
   → SAVED (R²: 0.6777)


KeyboardInterrupt: 